### ESTE SERIA NUESTRO SCRIPT DE BARRIDO, QUE SE EJECUTARIA EN LA NUBE, Y QUE LLAMARIA A NUESTRO SCRIPT DE ENTRENAMIENTO train.py

el siguiente script de ejemplo entrena un modelo de regresión logística mediante un argumento:
> --regularization  


para establecer el hiperparámetro de tasa de regularización y registra el métrica de precisión con el nombre Accuracy.

In [11]:
import argparse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import mlflow

# get regularization hyperparameter
parser = argparse.ArgumentParser()
parser.add_argument('--regularization', type=float, dest='reg_rate', default=0.01)
parser.add_argument('--training_data', type=str, dest='training_data')
args = parser.parse_args([]) # si no pones [] no se leen los valores por defecto
# args es el objeto que contiene los argumentos de la línea de comandos, 
# en este caso solo hay uno: --regularization
reg = args.reg_rate #esta variable extraera el valor de --regularization

# load the training dataset
data = pd.read_csv("./data/iris-data/iris.csv")

# separate features and labels, and split for training/validatiom
X = data.iloc[:, :-1].values #coje las columnas de la 0 a la penultima
#tambien podria ponerse X = data.iloc[:, 0:4].values #coje las columnas de la 0 a la 3
# o data[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].values
# o data.iloc[:, [0,1,2,3]].values
y = data.iloc[:, -1].values #coje la ultima columna
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30)

# train a logistic regression model with the reg hyperparameter
model = LogisticRegression(C=1/reg, solver="lbfgs", max_iter=1000).fit(X_train, y_train)
#he cambiado el solver de 'liblinear' a 'lbfgs' pq el primero no 
# soporta multinomial y el iris tiene 3 clases (3 valores de target - 3 tipos de flores)

# calculate and log accuracy
y_hat = model.predict(X_test)
acc = np.average(y_hat == y_test)
mlflow.log_metric("Accuracy", acc)

ESTE CODIGO LO TENEMOS QUE MODIFICARLO PARA QUE SE EJECUTE EN LA NUBE, Y CORRA PERFECTAMENTE.


[ARCHIVO MODIFICADO PARA QUE SE EJECUTE EN LA NUBE](./src/irish-src/train.py)


POR ESO TENEMOS QUE DEFINIR RUTAS RELATIVAS A LA NUBE, POR LO QUE USAREMOS EL ARGUMENT PARSER PARA PODER ESTABLER INDICACIONES NOTEBOOK - NUBE.

---
---

### Creamos trabajo Base
Para preparar el trabajo de barrido desde el notebook, primero debe crear un trabajo base de comando que especifique qué script se va a ejecutar y defina los parámetros usados por el script

El command job de Azure ML espera un script .py separado porque:

code="./src" → Azure ML sube todo esa carpeta a la nube

command="python train.py ..." → ejecuta ese script dentro del entorno

Prepara esto: un script train.py que recibe un --regularization y unos datos, en una máquina aml-cluster". Pero no lo ejecuta hasta que haces create_or_update.

In [ ]:
from azure.ai.ml import command
from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes

# configure command job as base
job = command(
    code="./src/irish-src",
    command="python train.py --training_data ${{inputs.training_data}} --regularization ${{inputs.reg_rate}}",
    inputs={
        "training_data": Input(
            type=AssetTypes.MLTABLE, 
            path="azureml:iris-dataset:1"),
        "reg_rate": 0.01,
    },
    environment="Mi_docker-Img:1",
    compute="cpu-cluster",
    )

*azureml:iris-dataset:1 lo añadi de AzurreML en practicas anteriores.


---
---

### Espacio de busqueda y Barrido de hiperparametros

In [45]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import os
from azure.ai.ml.sweep import Choice

#define el ESPACIO DE BUSQUEDA para el hiperparámetro reg_rate, 
command_job_for_sweep = job(
    reg_rate=Choice(values=[0.01, 0.1, 1]),
)

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=os.getenv("AZURE_SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("AZURE_RESOURCE_GROUP"),
    workspace_name=os.getenv("AZURE_WORKSPACE_NAME")
)

# Aplicar el BARRIDO DE PARAMETOS para obtener el sweep_job
sweep_job = command_job_for_sweep.sweep(
    compute="cpu-cluster",
    sampling_algorithm="grid",
    primary_metric="Accuracy",
    goal="Maximize",
)

# set the name of the sweep job experiment
sweep_job.experiment_name="sweep-iris-job"

# define the limits for this sweep
sweep_job.set_limits(max_total_trials=4, max_concurrent_trials=2, timeout=7200)

# submit the sweep
returned_sweep_job = ml_client.create_or_update(sweep_job)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
